# Income

GDP per capita at purchasing power parity from **[Kummu, Kosonen & Masoumzadeh Sayyar 2025 — Downscaled gridded global dataset for GDP per capita PPP over 1990–2022](https://www.nature.com/articles/s41597-025-04487-x)** (Aalto University, [Zenodo record 16741980](https://zenodo.org/records/16741980)). The 30 arc-min (0.5°) multiband GeoTIFF has one band per year 1990–2022 in 2017 international USD per person. It downscales national statistics onto admin-2 boundaries (~43 500 units, subnational data for 89 countries), so values are jurisdiction-flat inside admin units rather than truly gridded — matching `DATASETS.md`'s caveat that no true global gridded income source exists (🔴 Tier C).

We take the latest year (2022, band 33), snap the raster onto the atlas grid, `log10`-transform, and mask ocean via `is_land` from `grid.nc`. The log transform is essential because per-capita income spans ~2.5 orders of magnitude between the poorest (~$500) and richest (~$130 000) admin-2 units — a linear min-max at the scoring stage would collapse everything below the top decile. Higher income = better, so no sign inversion. Chose the 30 arcmin file over the 5 arcmin one because they encode the same admin-2-flat information and the 30 arcmin version already aligns to the atlas grid (4.8 MB vs 121 MB download).

In [ ]:
import numpy as np
import rasterio
import xarray as xr

from common import RAW_DIR, download, load_grid, plot_map, save_variable

VARIABLE = 'income'
variable_raw = RAW_DIR / VARIABLE
variable_raw.mkdir(parents=True, exist_ok=True)

YEAR = 2022  # latest year in the Kummu et al. 2025 release
BASE_YEAR = 1990  # band 1 = 1990, band N = 1989 + N

## 1. Fetch raw data

Single 30 arc-min multiband GeoTIFF from Zenodo (~5 MB). Cached under `variable_raw`; the download only runs once.

In [ ]:
URL = 'https://zenodo.org/records/16741980/files/rast_adm2_gdp_perCapita_1990_2022_30arcmin.tif?download=1'
tif_path = variable_raw / 'rast_adm2_gdp_perCapita_1990_2022_30arcmin.tif'

if not tif_path.exists():
    print(f'downloading {URL}')
    await download(URL, tif_path)

print(f'{tif_path.name}: {tif_path.stat().st_size / 1024**2:.1f} MB')

## 2. Read the latest-year band and snap to the atlas grid

Bands are 1-indexed and start at 1990, so `band = YEAR - BASE_YEAR + 1`. The source affine transform has negative pixel height (rows go north→south), giving a descending latitude coord — `xarray.interp` handles either direction as long as coords are monotonic. Nearest-neighbour resampling is appropriate because the underlying data is admin-2-flat: any smoothing across boundaries would blur real jurisdictional discontinuities into the score.

Non-positive or nodata pixels become `NaN` so they neither log-explode nor get counted as "very poor" — `weighted_score` at the top level renormalizes per-cell weights so a missing income neither drags nor lifts the surrounding livability.

In [ ]:
grid = load_grid()

with rasterio.open(tif_path) as src:
    assert src.crs.to_epsg() == 4326, f'expected EPSG:4326, got {src.crs}'
    band = YEAR - BASE_YEAR + 1
    assert 1 <= band <= src.count, f'band {band} outside available 1..{src.count}'
    raw = src.read(band).astype('float64')
    nodata = src.nodata
    src_lon = src.transform.c + src.transform.a * (np.arange(src.width) + 0.5)
    src_lat = src.transform.f + src.transform.e * (np.arange(src.height) + 0.5)

mask = np.isfinite(raw) & (raw > 0)
if nodata is not None:
    mask &= raw != nodata
raw = np.where(mask, raw, np.nan)

gdp_pc = xr.DataArray(
    raw,
    coords={'lat': src_lat, 'lon': src_lon},
    dims=('lat', 'lon'),
    name='gdp_per_capita_ppp',
).interp(lat=grid.lat, lon=grid.lon, method='nearest')

finite = gdp_pc.where(np.isfinite(gdp_pc))
print(f'{int(finite.notnull().sum())} of {gdp_pc.size} cells covered in {YEAR} '
      f'(range ${float(finite.min()):,.0f} – ${float(finite.max()):,.0f})')

## 3. Log-transform and mask ocean

Higher log-income = better, matching the atlas convention. Ocean cells are masked via `is_land`.

In [ ]:
values = (
    np.log10(gdp_pc)
    .astype('float32')
    .rename(VARIABLE)
    .where(grid.is_land == 1)
)
values.attrs['units'] = 'log₁₀(2017 international USD per person)'
values.attrs['source'] = f'Kummu et al. 2025 rast_adm2_gdp_perCapita_1990_2022_30arcmin.tif band {YEAR - BASE_YEAR + 1}'
values.attrs['source_year'] = YEAR
values

## 4. Plot

In [ ]:
plot_map(values, cmap='RdYlGn', robust=True)

## 5. Save

In [ ]:
out = save_variable(values, VARIABLE)
print(f'wrote {out}')